# When the past does not repeat

**Lecture 20 · Break → Fix** · Géron, Chapter 15

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**This notebook reproduces the broken measurement first.** Both numbers are
printed side by side before anything is repaired, because the point is not that
one of them is wrong — it is that nothing in the output of the wrong one tells
you so.

**A CPU runtime is enough**, and for sequences this short it is faster than the
GPU: the models are small enough that moving the data costs more than the
arithmetic saves.

**About the prompt boxes.** Where a code cell is preceded by a quoted prompt,
three lines follow it: what the prompt leaves open, the version a student
typically writes instead, and how you would catch a wrong answer. Those three
lines are the part worth reading twice.

The prompts here are **specifications, not transcripts** — this is what you
would have to ask for in order to get this cell, not a recording of somebody
asking for it. If your own prompt is vaguer than the box, expect worse code than
the cell below it.

*Lecture 19 is the one exception in this course.* It was built cell by cell
against Colab's Gemini 3.1 Pro, and its prompts are verbatim. It says so itself.


## 1 · Setup and the same data

> **Prompt**
>
> **input** · the same CTA file as the previous lecture
>
> **output** · the tidied frame and the 2016-2019 pool, ready to model
>
> **constraint** · identical preparation to Lecture 19, so any difference in the numbers is the protocol and not the data
>
> **check** · print the day count and confirm it matches

**Watch this prompt.**

* **Left open:** it says *identical* preparation but not how you would know it was identical. Copying cells by eye is exactly how two notebooks drift apart, and the drift shows up as a number you cannot attribute.
* **The usual student version:** re-typing the loader from memory, with one small improvement. The improvement is the problem: from here on, any difference against Lecture 19 could be the fix or could be the retyping, and you can no longer tell which.
* **How you would catch it:** print the pool length and date range and compare with Lecture 19's 1,247 days, 2016-01-01 to 2019-05-31. If either differs, stop here — nothing downstream is comparable until it matches.

In [ ]:
# --- setup -------------------------------------------------------------------
import sys
from pathlib import Path
import tarfile, urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print(f"python  {sys.version.split()[0]}")
print(f"torch   {torch.__version__}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

def load_ridership():
    tarball = Path("datasets/ridership.tgz")
    if not tarball.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(
            "https://github.com/ageron/data/raw/main/ridership.tgz", tarball)
        with tarfile.open(tarball) as t:
            t.extractall(path="datasets", filter="data")
    return pd.read_csv(
        "datasets/ridership/CTA_-_Ridership_-_Daily_Boarding_Totals.csv",
        parse_dates=["service_date"])

df = load_ridership()
df.columns = ["date", "day_type", "bus", "rail", "total"]
df = df.sort_values("date").set_index("date").drop("total", axis=1)
df = df.drop_duplicates()

WINDOW = 56
pool = df["rail"]["2016-01":"2019-05"]
print(f"{len(df):,} days on file; {len(pool):,} in the pool we model")

## 2 · Thread 10 — stationarity

A series is **strictly stationary** if the joint distribution of
$(X_{t_1},\dots,X_{t_k})$ is unchanged when every index is shifted by the same
$h$. That is far more than anyone can check, so in practice we ask for **weak
stationarity**: constant mean, constant variance, and an autocovariance
$\gamma(h) = \operatorname{Cov}(X_t, X_{t+h})$ that depends on the lag $h$ and
not on $t$.

**Why a model needs it.** Fitting one set of weights to all of history assumes
that what a value meant in 2016 is what it means in 2019. If the mean drifts,
the model is averaging two different worlds and is right about neither.

Ridership is not stationary: there is a downward trend and a hard weekly cycle.

> **Prompt**
>
> **input** · the rail pool, and the same series differenced at lag 1 and lag 7
>
> **output** · an augmented Dickey-Fuller p-value for each, with a verdict
>
> **constraint** · install statsmodels only if it is missing, so the notebook runs on a bare environment
>
> **check** · the level series should fail to look stationary where the differenced ones do not

**Watch this prompt.**

* **Left open:** what counts as a pass. ADF's null hypothesis is *non*-stationarity, so a **small** p-value is the stationary verdict. The direction is easy to invert and nothing in the printed output reminds you which way round it goes.
* **The usual student version:** reporting the test statistic with no critical values, or reading a large p as 'stationary'. A number with no stated direction is not a verdict, it is decoration.
* **How you would catch it:** decide before running which of the three you expect to fail. The raw series should not look stationary; the seasonal difference should. If all three pass, you have read the sign backwards — which is the single most common ADF error and it never raises anything.

In [ ]:
try:
    from statsmodels.tsa.stattools import adfuller
except ImportError:                         # Colab has it; a bare venv may not
    !pip -q install statsmodels
    from statsmodels.tsa.stattools import adfuller

def adf(series, label):
    stat, p, *_ = adfuller(series.dropna())
    verdict = "stationary" if p < 0.05 else "NOT stationary"
    print(f"{label:28s} ADF p = {p:7.4f}   {verdict}")

adf(pool,             "rail, as it is")
adf(pool.diff(),      "first difference")
adf(pool.diff(7),     "seasonal difference (7)")

### Differencing

$\nabla X_t = X_t - X_{t-1}$ removes a linear trend; applying it twice removes a
quadratic one. A series that is stationary after $d$ differences is *integrated
of order $d$*. And $\nabla_7 X_t = X_t - X_{t-7}$ removes a weekly cycle, which
is what this series most needs.

**Differencing is not free.** For a stationary series,

$$\operatorname{Var}(X_t - X_{t-h}) = 2\gamma(0)\,(1 - \rho(h))$$

so differencing at a lag where the autocorrelation $\rho(h)$ is *below* $1/2$
makes the variance **larger**, not smaller. Check it against the data rather than
believing it.

> **Prompt**
>
> **input** · the pool, and its autocorrelation at lags 1, 7 and 14
>
> **output** · predicted and measured standard deviation of each difference
>
> **constraint** · predict from the identity Var(X_t - X_t-h) = 2 gamma(0)(1 - rho(h)) BEFORE measuring, so the theory is exposed to the data
>
> **check** · flag any lag where differencing makes the spread larger, which is the point of the cell

**Watch this prompt.**

* **Left open:** it asks for predicted and measured but not for the **gap** between them. Two columns of plausible numbers is precisely where a 10% disagreement sits unnoticed.
* **The usual student version:** measuring only, and presenting the measurement as if it confirmed a formula nobody actually wrote down. The identity is what makes this a test rather than a description of what happened.
* **How you would catch it:** print predicted/measured as a ratio and expect 1.00 to within a per cent at every lag. If lag 7 agrees and lag 1 does not, suspect your estimate of rho, not the identity.

In [ ]:
sd = pool.std()
print(f"{'series':28s} sd {sd:>10,.0f}")
for h in (1, 7, 14):
    rho = pool.autocorr(lag=h)
    predicted = np.sqrt(2 * sd**2 * (1 - rho))
    measured = pool.diff(h).std()
    flag = "  <- WORSE than not differencing" if measured > sd else ""
    print(f"lag {h:>2d}: rho {rho:+.3f}   predicted sd {predicted:>10,.0f}"
          f"   measured {measured:>10,.0f}{flag}")

**Read the first row.** At lag 1 the autocorrelation is well under a half, so
first-differencing this series *increases* its spread. The textbook reflex —
"it is not stationary, difference it" — makes the problem harder here. At lag 7
the autocorrelation is high and the difference is genuinely smaller.

That is the whole explanation of why "copy last week" was so hard to beat in the
previous lecture: $\nabla_7$ is close to white noise, and a naive forecast is
exactly the model that assumes it *is*.

> **Prompt**
>
> **input** · five months of the rail series, raw and differenced at lags 1 and 7
>
> **output** · three stacked panels sharing an x axis
>
> **constraint** · a zero line on each, so 'bigger swings' is visible rather than asserted
>
> **check** · the first difference should look wilder than the series it came from

**Watch this prompt.**

* **Left open:** whether the three panels share a y-scale. They must **not** — the differenced series being smaller is the entire point — but a shared axis is the default in several plotting idioms and it flattens the bottom two panels into straight lines.
* **The usual student version:** three separate figures. The claim lives in the comparison between panels, and a reader who has to scroll between them is not comparing, they are remembering.
* **How you would catch it:** the zero line should be visibly crossed on the differenced panels and nowhere near the raw one. If all three look alike, the y-limits are shared and the figure is showing you nothing.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
recent = pool["2019-01":"2019-05"]
for ax, (s, title) in zip(axes, [
        (recent,          "the series"),
        (recent.diff(),   "first difference — bigger swings, not smaller"),
        (recent.diff(7),  "seasonal difference at lag 7 — nearly noise")]):
    ax.plot(s.index, s.values, lw=1, color="#0b3d62")
    ax.axhline(0, color="#c0392b", lw=1)
    ax.set_title(title, fontsize=11, loc="left")
plt.tight_layout(); plt.show()

### The autocorrelation function

One picture that contains everything above: correlation against lag. Spikes at
7, 14, 21 and a slow decay elsewhere.

> **Prompt**
>
> **input** · autocorrelation of the pool and of its seasonal difference, lags 0 to 42
>
> **output** · both on one bar chart
>
> **constraint** · plot them side by side at each lag, not on two charts, so the collapse at lag 7 is directly comparable
>
> **check** · spikes at 7, 14 and 21 in the raw series; nothing much left after differencing

**Watch this prompt.**

* **Left open:** what the collapse is supposed to look like. Name it before you run: the raw ACF stays high at every multiple of 7, the differenced one does not. A figure with no predicted shape cannot disappoint you.
* **The usual student version:** two separate charts, or only the differenced one. A collapse is a comparison; a single series cannot display it however clean it looks.
* **How you would catch it:** read lag 7 and lag 14 specifically and say the two numbers out loud. Those two bars are the claim. The other forty-one are scenery, and scanning all forty-three is how you talk yourself into a pattern.

In [ ]:
lags = np.arange(0, 43)
acf_level = [pool.autocorr(lag=int(k)) if k else 1.0 for k in lags]
acf_diff7 = [pool.diff(7).autocorr(lag=int(k)) if k else 1.0 for k in lags]

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(lags - 0.2, acf_level, width=0.4, label="the series", color="#0b3d62")
ax.bar(lags + 0.2, acf_diff7, width=0.4, label="after seasonal differencing",
       color="#14663a")
ax.axhline(0, color="#33414d", lw=1)
ax.set_xlabel("lag, in days"); ax.set_ylabel("autocorrelation")
ax.legend(); ax.set_title("Where the structure is")
plt.show()

## 3 · The broken measurement, reproduced

Exactly the cell from the previous lecture. Run it and look at the folds: they
agree with one another, which is what a stable measurement looks like.

> **Prompt**
>
> **input** · the pool as 56-lag windows
>
> **output** · a cross-validated MAE from a shuffled five-fold split
>
> **constraint** · this is the previous lecture's broken cell, reproduced exactly
>
> **check** · look at the fold spread — they agree with each other, which is what a stable measurement of the wrong quantity looks like

**Watch this prompt.**

* **Left open:** nothing, deliberately — this is Lecture 19's defective cell reproduced without changes. It is here because a fix you cannot put beside the original is not a demonstrated fix, it is a claim.
* **The usual student version:** skipping this cell on the grounds that we already know it is wrong. You then have no number to set the corrected ones against, and the rest of the section becomes assertion.
* **How you would catch it:** it must reproduce Lecture 19's figure to the boarding. If it does not, one of the two notebooks has drifted and every comparison below is void — go back and fix that before reading on.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

values = pool.values / 1e6
X = np.stack([values[i:i + WINDOW] for i in range(len(values) - WINDOW)])
y = values[WINDOW:]
print(f"X {X.shape}   y {y.shape}")

model = LinearRegression()
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds_random = -cross_val_score(model, X, y, cv=cv,
                                scoring="neg_mean_absolute_error") * 1e6
print(f"random 5-fold   MAE {folds_random.mean():>10,.0f}   "
      f"folds {np.round(folds_random).astype(int)}")

### The same model, split by time

`TimeSeriesSplit` never lets a training row come after a test row. One call
changes, nothing else.

> **Prompt**
>
> **input** · the same X and y
>
> **output** · the same five-fold MAE, split by time instead
>
> **constraint** · no training row may come after a test row; change one call and nothing else
>
> **check** · report the gap against the shuffled number in boardings and per cent

**Watch this prompt.**

* **Left open:** it says no training row may come after a test row, which is the right rule, but does not say to **verify** it. The guarantee comes from the row order, and the row order is an assumption inherited from three cells ago.
* **The usual student version:** `train_test_split(X, y, test_size=0.2)`. That function shuffles by default. It is the most common way this exact defect survives its own fix — you believe you have split by time and you have re-shuffled.
* **How you would catch it:** compare the largest training index with the smallest test index and assert the gap. One line, and it fails loudly the day someone sorts the frame upstream.

In [ ]:
cv = TimeSeriesSplit(n_splits=5)          # was KFold(shuffle=True)
folds_time = -cross_val_score(model, X, y, cv=cv,
                              scoring="neg_mean_absolute_error") * 1e6
print(f"forward 5-fold  MAE {folds_time.mean():>10,.0f}   "
      f"folds {np.round(folds_time).astype(int)}")
print()
print(f"the shuffle flattered the model by "
      f"{folds_time.mean() - folds_random.mean():,.0f} boardings "
      f"({100 * (folds_time.mean() - folds_random.mean()) / folds_time.mean():.0f}%)")

**Now look at the spread.** The forward folds disagree with each other far more
than the shuffled ones did — and that disagreement is real information, not
noise to be averaged away. The first fold trains on a couple of hundred days and
scores badly; the later folds train on years.

Two conditions have to hold for a split to be honest here, and the shuffle broke
both:

1. **No training row may come after a test row.** Otherwise the model has seen
   the future.
2. **No training row may be adjacent to a test row.** Two consecutive days are
   nearly the same number, so a neighbour in the training set is very close to
   giving away the answer.

`TimeSeriesSplit` fixes the first. The second needs a **gap**.

> **Prompt**
>
> **input** · the same X and y again
>
> **output** · MAE with a gap of one window between train and test
>
> **constraint** · no training row may be ADJACENT to a test row either — two consecutive days are nearly the same number
>
> **check** · print mean and fold spread for all three protocols together

**Watch this prompt.**

* **Left open:** how big the gap should be. One window is the defensible choice here because a window is exactly the reach of the leak — but the prompt does not say so, and a gap chosen because it improves the number is a hyperparameter tuned on the test set.
* **The usual student version:** assuming the time split already solved it. Adjacent days are nearly the same day; a test row one step after the last training row shares 55 of its 56 inputs with it.
* **How you would catch it:** the gapped number must be **worse** again than the plain time split. Each time you remove a route for information to travel, the score gets worse and more honest. A protocol that improves the score is a protocol you should distrust.

In [ ]:
# Condition 2, made explicit: leave a gap the width of one window between the
# end of training and the start of testing, so no test target can be predicted
# from a day that is effectively in the training set.
cv = TimeSeriesSplit(n_splits=5, gap=WINDOW)
folds_gap = -cross_val_score(model, X, y, cv=cv,
                             scoring="neg_mean_absolute_error") * 1e6
print(f"forward + gap   MAE {folds_gap.mean():>10,.0f}   "
      f"folds {np.round(folds_gap).astype(int)}")

for label, f in [("random 5-fold", folds_random),
                 ("forward", folds_time),
                 ("forward + purge", folds_gap)]:
    print(f"{label:18s} {f.mean():>10,.0f}   sd across folds {f.std():>9,.0f}")

### The margin, recomputed

The number that matters is not the MAE. It is how much of the naive baseline's
score the model actually takes off — and that is what the split was inflating.

> **Prompt**
>
> **input** · the naive baseline and the four protocols' MAEs
>
> **output** · the margin over the baseline that each protocol reports
>
> **constraint** · quote the margin, not the MAE — the margin is what the shuffle was inflating
>
> **check** · state what share of the claimed margin was protocol rather than model

**Watch this prompt.**

* **Left open:** it asks for margins rather than MAEs, which is right, but does not say the baseline must be scored on **each protocol's own test days**. Lecture 19 got this wrong and said so in its section 9.
* **The usual student version:** quoting the MAEs and letting the reader do the subtraction. The MAE moves when the test window moves; the margin is what the shuffle was inflating, so the margin is the number under discussion.
* **How you would catch it:** the margin should shrink monotonically as the protocol gets stricter. If it does not, either a protocol is mis-implemented or the baseline is being measured over a different set of days than the models.

In [ ]:
target = pool[WINDOW:]
naive = pool.shift(7)[WINDOW:]
mask = target.notna() & naive.notna()
NAIVE_MAE = float((target[mask] - naive[mask]).abs().mean())

# Four protocols, one model, one dataset. The only thing that changes is which
# rows are allowed to train on which other rows.
cut = int(len(X) * 0.8)
holdout = 1e6 * np.abs(
    LinearRegression().fit(X[:cut], y[:cut]).predict(X[cut:]) - y[cut:]).mean()

protocols = [("random 5-fold",           folds_random.mean()),
             ("one forward hold-out",    holdout),
             ("forward 5-fold",          folds_time.mean()),
             ("forward 5-fold + purge",  folds_gap.mean())]

print(f"naive baseline {NAIVE_MAE:,.0f}")
print()
print(f"{'protocol':26s}{'MAE':>10s}{'margin':>10s}")
for name, score in protocols:
    print(f"{name:26s}{score:>10,.0f}{(NAIVE_MAE - score) / NAIVE_MAE:>9.1%}")

claimed = 100 * (NAIVE_MAE - folds_random.mean()) / NAIVE_MAE
real    = 100 * (NAIVE_MAE - folds_gap.mean())    / NAIVE_MAE
print()
print(f"{100 * (claimed - real) / claimed:.0f}% of the claimed margin "
      f"was the protocol, not the model")

### Which of those four is "the" number?

The lecture quotes the **single forward hold-out**, because it is the protocol
that matches how the model would actually be used: fit once on everything up to
a date, forecast forward from there. The purged five-fold is stricter still, and
its margin is smaller again — mostly because its first fold trains on a couple of
hundred days and is scored anyway.

The useful discipline is not picking the smallest number. It is **saying which
protocol produced the one you quote**, so that a reader can reproduce it and a
colleague can disagree with it. A margin without a protocol attached is not a
result.

## 4 · Spending what is left

The honest margin is smaller, so the improvements have to be real. Three, in
order of how much they buy.

### Improvement 1 — more series, not more layers

Stacking three recurrent layers is the reflex, and on 1,191 windows it overfits.
What actually helps is giving the model something it does not already have: bus
ridership, and **tomorrow's day type**, which is known in advance from a calendar
and is therefore not a leak.

> **Prompt**
>
> **input** · rail, bus, and tomorrow's day type from the calendar
>
> **output** · a five-column frame, day type one-hot encoded
>
> **constraint** · shift(-1) on the CALENDAR is legitimate and shift(-1) on the target is a leak — the test is whether the value is knowable at prediction time
>
> **check** · assert the column names, so a silent change in encoding stops the notebook

**Watch this prompt.**

* **Left open:** why one `shift(-1)` is legitimate and another is a leak. The calendar is knowable tomorrow; the ridership is not. The prompt states the rule but the code cannot enforce it — the two shifts look identical.
* **The usual student version:** shifting everything by -1 for symmetry, or shifting nothing and wondering why the day-type column does not help. Both run.
* **How you would catch it:** for each shifted column ask: would I know this value at 6pm the day before? If yes it is a feature, if no it is the answer. That question is the whole of leak detection and it takes five seconds a column.

In [ ]:
mulvar = df[["rail", "bus"]] / 1e6
mulvar["next_day_type"] = df["day_type"].shift(-1)   # known in advance
mulvar = pd.get_dummies(mulvar, dtype=float)         # 5 columns

assert list(mulvar.columns) == ["rail", "bus", "next_day_type_A",
                                "next_day_type_U", "next_day_type_W"], \
    list(mulvar.columns)
mulvar = mulvar["2016-01":"2019-05"].dropna()
print(mulvar.shape, list(mulvar.columns))
mulvar.head(3)

**`shift(-1)` again — and this time it is legitimate.** In Lecture 19 a
`shift(-1)` on the *target* was a leak. Here it is applied to the calendar, and
the difference is not the sign: it is whether the value would be available at
the moment of the forecast. Tomorrow's day type is on a wall planner. Tomorrow's
ridership is not.

State the rule you are using, every time, in one line: *would I know this number
when I have to make the prediction?*

> **Prompt**
>
> **input** · the multivariate frame, a window and a horizon
>
> **output** · windows over all series, with rail alone as the target
>
> **constraint** · one windowing function used for every model below, so the comparison is like for like
>
> **check** · print the shapes and the train/test split sizes

**Watch this prompt.**

* **Left open:** that this function is about to be the single point of failure for every model below. An off-by-one here is not one wrong number, it is every comparison in the section wrong by the same amount — and consistently wrong looks exactly like correct.
* **The usual student version:** a second windowing function for the multivariate case, because the shapes differ. Two functions means the comparison measures the difference between the functions as well as between the models.
* **How you would catch it:** run it on a tiny array with distinct values and read the pairs by eye, as Lecture 19 did with six integers. Shapes agreeing is not the same as contents aligning, and only one of the two is checkable at scale.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def make_windows(frame, window=WINDOW, horizon=1):
    """Windows over several series; the target is `rail` only."""
    arr = torch.tensor(frame.values, dtype=torch.float32)
    rail = arr[:, 0]
    n = len(arr) - window - horizon + 1
    Xs = torch.stack([arr[i:i + window] for i in range(n)])
    ys = torch.stack([rail[i + window:i + window + horizon] for i in range(n)])
    return Xs, ys

Xm, ym = make_windows(mulvar)
cut = int(len(Xm) * 0.8)
print(f"X {tuple(Xm.shape)}   y {tuple(ym.shape)}   train {cut}, test {len(Xm) - cut}")

### Improvement 2 — gates

A simple RNN multiplies by the same recurrent matrix at every step, so gradients
over 56 steps either vanish or explode — Lecture 13's thread, in a new place. A
**GRU** adds two gates: an update gate that decides how much of the old state to
keep, and a reset gate that decides how much of it to use. Keeping is now
addition rather than repeated multiplication, so a gradient can travel.

> **Prompt**
>
> **input** · windows of five series
>
> **output** · a trained GRU and its held-out MAE
>
> **constraint** · gates rather than a plain RNN, because a 56-step recurrence multiplies by the same matrix every step
>
> **check** · watch the held-out MAE as it trains rather than reading only the final number

**Watch this prompt.**

* **Left open:** what to do if it fails to converge. A GRU on 950 rows can sit at its initialisation for fifty epochs and then move; without a per-epoch print you cannot tell that from a dead model.
* **The usual student version:** switching to a GRU and reporting the improvement, without noticing that the series count changed at the same time. Two changes, one number, nothing learned — which is exactly what the next cell is for.
* **How you would catch it:** the held-out MAE must fall and then flatten. If it rises, say so and report the final epoch, not the best one you saw. Lecture 19's RNN rose at epoch 160 and the honest number was the one at 200.

In [ ]:
class GruModel(nn.Module):
    def __init__(self, input_size=5, hidden_size=32, output_size=1):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, _state = self.rnn(X)
        return self.head(outputs[:, -1])

def train(model, Xtr, ytr, Xte, yte, epochs=120, lr=0.005, quiet=False):
    loss_fn = nn.HuberLoss(delta=0.05)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
    for epoch in range(epochs):
        model.train()
        for Xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(Xb), yb).backward()
            opt.step()
        if not quiet and ((epoch + 1) % 30 == 0 or epoch == 0):
            model.eval()
            with torch.no_grad():
                v = 1e6 * (model(Xte) - yte).abs().mean().item()
            print(f"  epoch {epoch + 1:>3d}   held-out MAE {v:>10,.0f}")
    model.eval()
    with torch.no_grad():
        return 1e6 * (model(Xte) - yte).abs().mean().item()

torch.manual_seed(RANDOM_STATE)
print("GRU on five series:")
gru_mae = train(GruModel(input_size=5), Xm[:cut], ym[:cut], Xm[cut:], ym[cut:])

> **Prompt**
>
> **input** · the same GRU, on rail alone
>
> **output** · its held-out MAE, beside the five-series version
>
> **constraint** · change ONE thing — two changes at once is not an experiment
>
> **check** · the table separates 'gates helped' from 'more series helped'

**Watch this prompt.**

* **Left open:** nothing — 'change ONE thing' is the constraint and it is the whole cell. The prompt is short because the discipline is simple to state and almost impossible to keep once you are curious.
* **The usual student version:** changing the architecture and the inputs together, then attributing the whole gain to whichever one they found more interesting. This is not dishonesty, it is impatience, and it produces the same result.
* **How you would catch it:** you should be able to name, in one sentence, the single difference between this run and the previous one. If the sentence needs an 'and', it is two experiments and it answers neither.

In [ ]:
# The same model on rail alone, to separate "gates helped" from "more series
# helped". Two changes at once is not an experiment.
torch.manual_seed(RANDOM_STATE)
Xr, yr = make_windows(mulvar[["rail"]])
print("GRU on rail alone:")
gru_rail = train(GruModel(input_size=1), Xr[:cut], yr[:cut], Xr[cut:], yr[cut:],
                 quiet=True)

print()
print(f"{'model':34s}{'MAE':>12s}{'vs naive':>11s}")
for name, score in [("copy last week", NAIVE_MAE),
                    ("linear, forward + purge", folds_gap.mean()),
                    ("GRU, rail only", gru_rail),
                    ("GRU, five series", gru_mae)]:
    print(f"{name:34s}{score:>12,.0f}{(NAIVE_MAE - score) / NAIVE_MAE:>10.1%}")

### Improvement 3 — forecast a fortnight, not a day

A staffing decision needs more than tomorrow. Two changes: the target becomes 14
values instead of 1, and the head produces 14 numbers.

> **Prompt**
>
> **input** · the same windows, with a fourteen-day target
>
> **output** · MAE at each horizon, plotted against the naive baseline
>
> **constraint** · one head producing fourteen numbers, not fourteen models
>
> **check** · read the SHAPE — a single average would hide that day 14 is no better than copying last week

**Watch this prompt.**

* **Left open:** how the naive baseline extends to fourteen days. Copying last week is well defined at horizon 7 and needs a decision at horizon 14 — and if you do not make it deliberately, the comparison silently changes shape halfway along the x axis.
* **The usual student version:** fourteen separate models, one per horizon. It runs, it scores better, and it cannot be deployed — you would be fitting fourteen things to answer one question, and each on less data than the last.
* **How you would catch it:** error must grow with horizon. If day 14 is predicted as accurately as day 1, you have leaked the future in: check that the target window starts after the input window ends, not at it.

In [ ]:
HORIZON = 14
Xh, yh = make_windows(mulvar, horizon=HORIZON)
cut_h = int(len(Xh) * 0.8)

torch.manual_seed(RANDOM_STATE)
horizon_model = GruModel(input_size=5, output_size=HORIZON)
h_mae = train(horizon_model, Xh[:cut_h], yh[:cut_h], Xh[cut_h:], yh[cut_h:],
              quiet=True)

horizon_model.eval()
with torch.no_grad():
    pred = horizon_model(Xh[cut_h:])
per_step = 1e6 * (pred - yh[cut_h:]).abs().mean(dim=0).numpy()

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(range(1, HORIZON + 1), per_step, marker="o", color="#0b3d62",
        label="the model")
ax.axhline(NAIVE_MAE, color="#c0392b", ls="--", label="copy last week")
ax.set_xlabel("days ahead"); ax.set_ylabel("MAE")
ax.set_title("Error against horizon — the margin is spent by about day 7")
ax.legend(); plt.show()

for k in (1, 7, 14):
    print(f"{k:>2d} days ahead   MAE {per_step[k - 1]:>10,.0f}"
          f"   vs naive {(NAIVE_MAE - per_step[k - 1]) / NAIVE_MAE:>7.1%}")

**Read the shape, not the average.** One number for "the fourteen-day forecast"
would hide that day 1 is good and day 14 is no better than copying last week.
Report the curve. If a decision only needs three days, say so and be judged on
three.

## 5 · Regime change

Everything above stops at May 2019. The series does not — in March 2020 the
level falls by roughly three quarters and never returns to where it was.

This is worth being precise about, because it is **not a leak and not a bug**.
The protocol was correct, the measurement was honest, and the model is still
useless afterwards. No split protects you from the world changing.

> **Prompt**
>
> **input** · the whole series, through 2021
>
> **output** · the mean level before and after March 2020, and a plot spanning both
>
> **constraint** · this is not a leak and not a bug — say so plainly; the protocol was right and the model still stopped working
>
> **check** · the ratio of the two levels is printed, not described

**Watch this prompt.**

* **Left open:** what you are entitled to conclude. This is the one place in the application where the model is not at fault and neither is the protocol — and 'not a bug' is a finding that has to be stated, because silence reads as an oversight.
* **The usual student version:** treating the 2020 collapse as a modelling failure and reaching for a bigger network. No amount of capacity recovers a distribution that stopped existing.
* **How you would catch it:** print the mean level either side of March 2020. If the two differ by more than the model's entire error budget, the question is not 'why is the model wrong' but 'when did this model stop being about the world'.

In [ ]:
level_2019 = df["rail"]["2019-01":"2019-05"].mean()
level_2020 = df["rail"]["2020-04":"2020-08"].mean()
print(f"mean daily rail boardings, early 2019   {level_2019:>10,.0f}")
print(f"mean daily rail boardings, mid 2020     {level_2020:>10,.0f}")
print(f"                                        {level_2020 / level_2019:>10.1%} "
      f"of the earlier level")

fig, ax = plt.subplots(figsize=(11, 3.2))
df["rail"]["2019-01":"2021-06"].plot(ax=ax, lw=0.8, color="#0b3d62")
ax.axvspan(pd.Timestamp("2020-03-15"), pd.Timestamp("2020-06-01"),
           color="#c0392b", alpha=0.15)
ax.set_title("A correct protocol, an honest number, and a model that stopped working")
plt.show()

**What to do about it** is a monitoring question, not a modelling one: measure
the live error against the committed number, and have a rule that says when to
stop trusting the model. A model that is never re-measured after deployment is
an assumption wearing a number's clothes.

## 6 · The temporal checklist

Take this to any dataset with a timestamp in it:

1. **Is there a time column?** If yes, no shuffled split — ever, including
   inside `cross_val_score`, `train_test_split` and any tuner's own CV.
2. **Is there a gap between train and test?** Adjacent rows leak.
3. **Would I know every feature at prediction time?** Say it out loud for each
   one. `shift(-1)` on a calendar is fine; on the target it is the answer.
4. **Is the baseline seasonal?** Compare against copying the same weekday, not
   against the mean.
5. **Did I difference reflexively?** Check $\rho(h) > 1/2$ first, or the
   variance goes up.
6. **Is the score a single number when the decision needs a curve?**
7. **What would tell me the regime has changed?** Write the trigger down before
   deployment, not after.

### ★ Record your numbers

The claimed margin, the honest margin, and the difference between them. That
difference is the most useful number in this application: it is the size of the
mistake that nothing in the output complained about.

### Red-team your own notebook

* Set `gap=0` in `TimeSeriesSplit`. How much of the margin comes back? That
  amount was adjacency.
* Give the model `df["day_type"]` **without** the `shift(-1)`. The score barely
  moves — explain why that is worse, not better.
* Train on 2016–2019, test on 2020. Then argue, in two sentences, whether the
  model was wrong or the question was.